# AlphaTrain Iteration 1: Self-Play Training (Colab)

Train the ResNet on 277K self-play states (493 games, 800 sims MCTS).
Policy learns MCTS visit distributions (KL-divergence). Value learns TD returns (MSE).

**Requires A100 runtime** (~10 GB VRAM for data + model). T4 may be too tight.

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code
2. `selfplay_iter1.pt` — training data (8.9 GB)
3. `alphatrain_td_best.pt` — base model to fine-tune

**Build tarball locally:**
```bash
tar czf colorlines_selfplay_train.tar.gz \
    --exclude='__pycache__' --exclude='*.nbc' --exclude='*.nbi' \
    --exclude='data' --exclude='.venv' --exclude='.git' \
    --exclude='*.tar.gz' --exclude='selfplay_data' \
    -C /Users/andreis/local/source/colorlines98 \
    alphatrain game
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil

DRIVE = '/content/drive/MyDrive/alphatrain'

# Extract code
!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

# Copy data + model
os.makedirs('/content/alphatrain/data', exist_ok=True)
for f in ['selfplay_iter1.pt', 'alphatrain_td_best.pt']:
    src = os.path.join(DRIVE, f)
    dst = os.path.join('/content/alphatrain/data', f)
    if not os.path.exists(dst):
        print(f'Copying {f}...')
        shutil.copy(src, dst)
    print(f'{f}: {os.path.getsize(dst)/1e6:.0f} MB')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')
    if g.total_memory < 20e9:
        print('WARNING: T4 may not have enough VRAM. Use A100 runtime.')

!cd /content && python -m pytest alphatrain/tests/test_model.py alphatrain/tests/test_observation.py -v --tb=short 2>&1 | tail -10

In [ ]:
# Iteration 1: self-play training
# Policy: KL-divergence with MCTS visit distributions
# Value: MSE on scalar TD returns (sigmoid * 500)
# Warm start from epoch 6 heuristic model
%cd /content
!python -m alphatrain.train \
    --tensor-file alphatrain/data/selfplay_iter1.pt \
    --gpu-data --amp --compile \
    --value-bins 1 --val-weight 1.0 \
    --resume alphatrain/data/alphatrain_td_best.pt --warm-start \
    --epochs 10 --batch-size 8192 --lr 3e-4 --warmup-epochs 2 \
    --copy-to /content/drive/MyDrive/alphatrain/selfplay_iter1_best.pt \
    --save-dir /content/checkpoints/selfplay_iter1

In [ ]:
# Evaluate: standalone policy score (baseline was 314)
%cd /content
!python -m alphatrain.evaluate --player policy \
    --model /content/checkpoints/selfplay_iter1/best.pt \
    --games 50 --seed 42

In [ ]:
# Copy final model to Drive
import shutil
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/selfplay_iter1/{f}'
    dst = f'{DRIVE}/selfplay_iter1_{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Saved {dst}')